#### RAG(Retrieval-Augmented Generation) 처리 과정
1. 문서 로드(Document Loading)
- PDF, TXT, 웹 문서 등의 데이터를 읽어온다.
2. 문서 파싱 및 청크 분할(Document Parsing & Chunking)
- 문서를 LLM이 처리할 수 있는 크기로 분할하고 전처리한다.
3. 임베딩 생성 및 벡터 데이터베이스 저장(Embedding & Vector Store)
- 문서 청크를 임베딩 벡터로 변환한 뒤 벡터 데이터베이스에 저장한다.
4. 사용자 질문 기반 유사도 검색(Similarity Search)
- 사용자의 질문과 의미적으로 유사한 문서를 벡터 데이터베이스에서 검색한다.
1. 검색된 문맥과 함께 LLM에 질의(Context Augmented Generation)
- 검색된 문서를 질문과 함께 LLM에 전달하여 보다 정확한 응답을 생성한다.

#### install langchain-community

In [ ]:
# %pip install docx2txt 
%pip install -U langchain-community

#### document load 

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader 
loader = Docx2txtLoader('./test.docx')
document = loader.load()

In [ ]:
document

#### Text Spliter 
- Character Spliter 
  - 하나의 구분자(separator) 기반으로 텍스트를 분할한다.
  - 단순 문자열 기준으로 분리하기 때문에 구조를 충분히 보존하지 못할 수 있다.
  - 비교적 단순한 텍스트 처리에 적합하다.
- Recursively Spliter
  - 여러 구분자를 우선순위 기반으로 순차적으로 적용한다.
  - 문단 → 줄바꿈 → 공백 순으로 재귀적으로 분할하여 문맥 손실을 최소화한다.
  - LangChain에서 가장 많이 사용되는 Text Splitter 중 하나이다.
  - RAG 환경에서 일반적으로 권장된다.

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 

text_spliter = RecursiveCharacterTextSplitter(
    chunk_size=1500,   #
    chunk_overlap=200, # 유사도 검색을 위해 확률을 높인다. 
)

loader = Docx2txtLoader('./test.docx')
document = loader.load_and_split(text_spliter)

In [ ]:
len(document)

#### Embedding OpenAI 

In [ ]:

# from langchain_openapi import OpenAIEmbeddings
# from dotenv import load_dotenv
# import os 
#load_dotenv(dotenv_path='./.env')
# openai_embedding = OpenAIEmbeddings() 

#### Embedding Ollama 
- ollama pull nomic-embed-text


In [ ]:

from langchain_ollama import OllamaEmbeddings
embedding = OllamaEmbeddings(
    model="nomic-embed-text"
)

#### install Pinecone

In [ ]:
%pip install --upgrade --quiet langchain-pinecone langchain-openai langchain pinecone-notebooks 

In [ ]:
import os 
import time 
import getpass
from pinecone import Pinecone, ServerlessSpec 
from dotenv import load_dotenv
from pinecone import Pinecone


load_dotenv()

if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass.getpass("Enter your Pinecone API key: ")
    print()
pinecone_api_key = os.environ.get("PINECONE_API_KEY")

pc = Pinecone(api_key=pinecone_api_key)

In [ ]:
from langchain_pinecone import PineconeVectorStore 

index_name = 'tax-index'
database = PineconeVectorStore.from_documents(document, embedding, index_name=index_name)

In [ ]:
query = '가상 자산 소득세는 얼마인가요?'
retrieved_docs = database.similarity_search(query, k=3)

In [ ]:
retrieved_docs

In [ ]:
from langchain_ollama import ChatOllama 

llm = ChatOllama(model="gemma4:latest") 

In [ ]:
prompt = f"""[Identity]
- 당신은 최고의 한국 소득세 전문가입니다
- [Context] 를 참고해서 사용자의 질문에 대답해주세요

[Context]
{retrieved_docs}

Question: 
{query}
"""

In [ ]:
ai_message = llm.invoke(prompt)
print(ai_message)

In [ ]:
ai_message.content

#### install langchain hub 

In [ ]:
# %pip install -U langchainhub --quiet
# %pip install -U langchain
%pip install langchainhub

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

template = """
[Identity]
당신은 최고의 한국 소득세 전문가입니다. 
[Context]
{context}

[Question]
{question}
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

In [ ]:
prompt

In [ ]:
%pip install -U langchain-community langchain

In [ ]:
from langchain_classic.chains import RetrievalQA 

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=database.as_retriever(),
    chain_type_kwargs={"prompt":
        prompt}
)

In [ ]:
ai_message = qa_chain({"query": query})

In [ ]:
ai_message